In [19]:
import pandas as pd

df = pd.read_csv('data/encykorea.csv')

df.columns

Index(['page', 'card_index', 'category', 'title', 'href', 'summary',
       'contents'],
      dtype='object')

In [20]:
# 불필요한 컬럼 제거
df = df.drop(columns=['page','card_index','href'])

In [21]:
# # ['title', 'summary', 'content'] 각 컬럼 내 텍스트 큰따옴표로 감싸기

# df_wrapped = df.copy()
# df_wrapped["title"] = df_wrapped["title"].astype(str).apply(lambda x: f'"{x.replace("\"", "\"\"")}"')
# df_wrapped["summary"] = df_wrapped["summary"].astype(str).apply(lambda x: f'"{x.replace("\"", "\"\"")}"')
# df_wrapped["content"] = df_wrapped["content"].astype(str).apply(lambda x: f'"{x.replace("\"", "\"\"")}"')

In [22]:
# 결측치 처리 및 문자열 변환
text_cols = ["title", "summary", "contents"]

# 기본 전처리 (NaN → 빈 문자열, 공백 정리)
for c in text_cols:
    df[c] = df[c].fillna("").astype(str).str.replace("\n", " ").str.replace("\r", " ")
    df[c] = df[c].str.replace(r"\s+", " ", regex=True).str.strip()

In [23]:
# 따옴표 정규화
def normalize_quotes(x):
    return x.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
for c in text_cols:
    df[c] = df[c].apply(normalize_quotes)

In [ ]:
# import re

# # content_clean 생성 (한자 제거)
# HANJA_REGEX = r"[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF]"
# EMPTY_PAREN_REGEX = r"\(\s*\)"

# def remove_hanja_safely(text):
#     if not isinstance(text, str):
#         return ""
#     # 1) 한자 제거
#     t = re.sub(HANJA_REGEX, "", text)
#     # 2) 괄호 안이 완전히 공백이 된 경우 → 괄호 자체 제거
#     t = re.sub(EMPTY_PAREN_REGEX, "", t)
#     # 3) 양쪽 공백 정리
#     return t.strip()

# df["title_clean"] = df["title"].apply(remove_hanja_safely)
# df["summary_clean"] = df["summary"].apply(remove_hanja_safely)
# df["content_clean"] = df["contents"].apply(remove_hanja_safely)

In [24]:
df.head(4)

,category,title,summary,contents
0,물품,가(枷),죄수의 목에 채우는 형구(刑具).,보통 '칼'이라고 하였다. 칼은 마른 나무널판으로 만든 형틀로 죄수의 목에 씌워 보...
1,문헌,20공신회맹축 - 보사공신녹훈후(二十功臣會盟軸 - 保社功臣錄勳後),1694년 갑술환국으로 재집권한 서인이 보사공신 회맹 때의 회맹문과 복훈 때의 축문...,숙종대에 갑술환국(甲戌換局)으로 서인(西人)이 재집권하면서 보사공신(保社功臣) 중 ...
2,문헌,20공신회맹축 - 영국공신녹훈후(二十功臣會盟軸 - 寧國功臣錄勳後),"1646년 9월, 조선 제16대 왕 인조가 영국공신(寧國功臣)을 녹훈한 이후에 20...",이십공신회맹축(二十功臣會盟軸)은 1646년(인조 24)에 영국공신(寧國功臣)을 녹훈...
3,제도,가감역관(假監役官),조선 후기 선공감의 종9품 임시관직.,"1718년(숙종 44)에 처음으로 설치하였고, 정원은 3인이다. 이들은 감역관과 함..."


In [28]:
import csv

df.to_csv('data/encykorea_cleaned.csv', index=False, quoting=csv.QUOTE_ALL)

### 기존 뜻 요약 및 데이터에서 단어 추출 후 뜻 작성

In [1]:
import pandas as pd
import re

df = pd.read_csv("data/encykorea_cleaned.csv")
df.head(2)

,category,title,summary,contents
0,물품,가(枷),죄수의 목에 채우는 형구(刑具).,보통 '칼'이라고 하였다. 칼은 마른 나무널판으로 만든 형틀로 죄수의 목에 씌워 보...
1,문헌,20공신회맹축 - 보사공신녹훈후(二十功臣會盟軸 - 保社功臣錄勳後),1694년 갑술환국으로 재집권한 서인이 보사공신 회맹 때의 회맹문과 복훈 때의 축문...,숙종대에 갑술환국(甲戌換局)으로 서인(西人)이 재집권하면서 보사공신(保社功臣) 중 ...


In [2]:

# 괄호(내부 내용까지) 제거, 한자 제거

def remove_parentheses_and_hanja(text):
    if not isinstance(text, str):
        return ""
    # 괄호와 내부 내용 제거
    text = re.sub(r"\([^)]*\)", "", text)
    # 한자 제거
    text = re.sub(r"[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF]", "", text)
    # 양쪽 공백 정리
    return text.strip()

for c in ["title", "summary", "contents"]:
    df[c] = df[c].apply(remove_parentheses_and_hanja)

df.head(2)

,category,title,summary,contents
0,물품,가,죄수의 목에 채우는 형구.,보통 '칼'이라고 하였다. 칼은 마른 나무널판으로 만든 형틀로 죄수의 목에 씌워 보...
1,문헌,20공신회맹축 - 보사공신녹훈후,1694년 갑술환국으로 재집권한 서인이 보사공신 회맹 때의 회맹문과 복훈 때의 축문...,숙종대에 갑술환국으로 서인이 재집권하면서 보사공신 중 1689년 기사환국으로 파훈되...


In [3]:
# OpenAI API를 사용하여 contents 컬럼에서 단어 추출 후 title에 추가
import json
from openai import OpenAI

client = OpenAI()

def extract_keywords_from_contents(title, summary, contents):
    if not isinstance(contents, str) or not contents.strip():
        return []
    
    # OpenAI API 호출을 위한 프롬프트 설정
    prompt_template = """
            You are helping to build a Korean history glossary for elementary school students.

            Goal:
                From the given Korean-history text, extract difficult terms for an average Korean elementary school student (age 10–12), and write simple definitions for each term.

            Rules:
                1. Do NOT include the main term given in "TITLE" as an extracted term.
                2. Extract only important historical or conceptual terms from the CONTENTS text
                   that are likely to be difficult for elementary students
                   (for example: 한자어, 역사 용어, 행정 단위, 제도 이름 등).
                3. Do NOT include very common words (예: 사람, 나라, 전쟁, 생활, 도움 등).
                4. For each extracted term, write a short definition in EASY KOREAN
                   that an elementary school student can understand.
                5. The definition should be 1–2 sentences, no more than 40 Korean characters if possible.
                6. If there are no suitable difficult terms, return an empty list.
                7. Output MUST be valid JSON in the following format:

            {{
                "extra_terms": [
                    {{
                    "term": "...",
                    "definition": "..."
                    }}
                ]
            }}

            Input:
            TITLE: "{title}"
            SUMMARY: "{summary}"
            CONTENTS: "{contents}"
            """
    prompt = prompt_template.format(
        title=title.replace('"', '\\"'),
        summary=summary.replace('"', '\\"') if isinstance(summary, str) else "",
        contents=contents.replace('"', '\\"'),
    )
    
    # OpenAI Chat Completions API 호출
    response = client.chat.completions.create(
        model="gpt-5-nano",
        # 가능하면 JSON 강제 (선택)
        # response_format={"type": "json_object"},
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    # 모델이 생성한 텍스트(JSON 문자열)를 추출
    if (
        not response.choices 
        or not response.choices[0].message 
        or not response.choices[0].message.content
    ):
        print("Empty response:", response)
        return []
    raw_output = response.choices[0].message.content.strip()

    # JSON 파싱
    try:
        data = json.loads(raw_output)
        extra_terms = data.get("extra_terms", [])
        # 리스트가 아닐 경우 방어 코드
        if not isinstance(extra_terms, list):
            return []
        return extra_terms
    except json.JSONDecodeError:
        # JSON 형식이 조금 틀어졌을 때를 대비한 예외 처리
        print("JSONDecodeError, raw_output:", raw_output)
        return []


In [4]:
import re

def clean_definition(text: str) -> str:
    """정의 끝의 '~에요', '~이에요', '~입니다', '~예요' 등을 잘라내기."""
    if not isinstance(text, str):
        return ""
    text = text.strip()
    # 끝부분 어미 제거
    text = re.sub(r"(이에요|에요|입니다|예요)\s*[\.\!]*$", "", text)
    return text

def update_row_with_terms(row):
    """
    한 행에서:
      - 추출된 용어 리스트를 반환.
      - 원래 title/summary는 수정하지 않음.
    반환 형식: [(term, definition), ...]
    """
    extra_terms = extract_keywords_from_contents(
        title=row["title"],
        summary=row.get("summary", "") or "",
        contents=row["contents"],
    )

    existing_title = str(row["title"])
    term_defs = []

    for t in extra_terms:
        term = (t.get("term") or "").strip()
        definition = clean_definition(t.get("definition", ""))

        # term이 없거나, 원래 title에 이미 포함된 용어는 제외
        if not term or term in existing_title:
            continue

        if not definition:
            continue

        term_defs.append((term, definition))

    return term_defs


In [5]:
from tqdm import tqdm
import csv
import os
from time import sleep

output_path = "output/result.csv"
start_idx = 5304  # 0-based 인덱스 기준 5304행부터 처리

# 파일 존재 여부 확인
file_exists = os.path.exists(output_path)

# output 폴더 생성 (없을 경우)
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# 파일이 없을 때만 헤더 기록
if not file_exists:
    with open(output_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "title", "summary"])

batch = []
batch_size = 20

# 처리 구간만 슬라이스
df_to_process = df.iloc[start_idx:]

# tqdm 추가 (슬라이스 길이)
for idx, row in tqdm(df_to_process.iterrows(), total=len(df_to_process), desc="Processing rows"):
    # 원본 df의 인덱스 사용(슬라이스여도 원래 인덱스 유지)
    row_id = idx

    # 1) 먼저 원래 항목을 그대로 한 줄 기록
    original_title = row["title"]
    original_summary = row.get("summary", "") or ""
    batch.append([row_id, original_title, original_summary])

    try:
        # 2) 추출된 용어들을 각각 별도의 행으로 추가
        term_defs = update_row_with_terms(row)  # [(term, definition), ...]

        for term, definition in term_defs:
            batch.append([row_id, term, definition])

    except Exception as e:
        print(f"Error at row {idx}: {e}")
        # 원본 행은 이미 batch에 들어갔으므로 그대로 진행
        pass

    # batch 기준으로 주기적으로 CSV에 append
    if len(batch) >= batch_size:
        with open(output_path, "a", newline="", encoding="utf-8-sig") as f:
            writer = csv.writer(f)
            writer.writerows(batch)
        batch = []
        sleep(0.2)  # rate limit 완화용

# 마지막 batch 저장
if batch:
    with open(output_path, "a", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)
        writer.writerows(batch)


Processing rows:  36%|███▌      | 1793/5040 [19:25:37<231:23:39, 256.55s/it]

Error at row 7096: Connection error.


Processing rows:  36%|███▌      | 1794/5040 [19:25:38<162:18:07, 180.00s/it]

Error at row 7097: Connection error.


Processing rows:  36%|███▌      | 1795/5040 [19:25:40<113:56:55, 126.41s/it]

...

Processing rows:  91%|█████████ | 4591/5040 [58:28:15<70:05:53, 562.03s/it] 

Error at row 9894: Connection error.


Processing rows:  91%|█████████ | 4592/5040 [58:28:16<49:00:30, 393.82s/it]

Error at row 9895: Connection error.


Processing rows: 100%|██████████| 5040/5040 [64:45:39<00:00, 46.26s/it]    


In [18]:
# 없는 행 확인
import pandas as pd

# 1) 컬럼명이 없는 CSV 읽기
df = pd.read_csv("output/result.csv", header=None, names=["id", "term", "description"])

# 2) expected 범위 정의
expected_ids = set(range(0, 10343))

actual_ids = set(df['id'].unique())
missing_ids = expected_ids - actual_ids

print("Missing IDs:", sorted(missing_ids))


Missing IDs: []


In [19]:
df.to_csv("output/result_edited1.csv", index=False)

In [20]:
df['term'].value_counts()

term
본관      1954
시호      1202
병과      1110
급제      1080
임진왜란     931
        ... 
전발무사       1
상형추의       1
의율차례       1
비상전초       1
경사요의       1
Name: count, Length: 33947, dtype: int64

In [14]:
df[df['id']==7147]

,id,term,description
91948,7147,이천기,"조선 후기에, 승지, 한림, 삼사이랑 등을 역임한 문신."


In [15]:
df['term'].nunique()

33947

In [16]:
df.shape[0]

131641

In [9]:
print("원본 길이 :",df.shape[0])
# [term, description] 중복 제거
df  = df.drop_duplicates(subset=["term", "description"]).reset_index(drop=True)
print("중복 제거 후 길이 :", df.shape[0])

원본 길이 : 131641
중복 제거 후 길이 : 125258


In [24]:
# 하나 이상인 것
term_counts = df['term'].value_counts()

# 두 번 이상 등장하는 term만
duplicated_terms = term_counts[term_counts > 1]

print(duplicated_terms)

term
본관      1954
시호      1202
병과      1110
급제      1080
임진왜란     931
        ... 
충무         2
폐습         2
춘추관사       2
석비         2
홍위         2
Name: count, Length: 9157, dtype: int64


In [25]:
print("두 번 이상 등장하는 term 종류 수 :", (term_counts > 1).sum())

두 번 이상 등장하는 term 종류 수 : 9157


In [26]:
print("두 번 이상 등장하는 term이 차지하는 행 수 :", term_counts[term_counts > 1].sum())

두 번 이상 등장하는 term이 차지하는 행 수 : 106851


In [40]:

# term 에서 중복 제거
# description 길이 계산
df["desc_len"] = df["description"].str.len()

# term별로 desc_len이 가장 긴 행만 남기기
df_unique_term = (
    df.sort_values(["term", "desc_len"], ascending=[True, False])
      .drop_duplicates(subset=["term"], keep="first")
      .drop(columns=["desc_len"])
      .reset_index(drop=True)
)

print("term 기준 유니크 개수:", df_unique_term["term"].nunique())
df_unique_term.to_csv("output/result_term_dedup_longest.csv", index=False)

term 기준 유니크 개수: 33947


In [41]:
df_unique_term[df_unique_term["term"]=="가(枷)"]

,id,term,description
395,0,가(枷),죄수의 목에 채우는 형구(刑具).


In [42]:
# 한자 제거 
import re
def remove_hanja(text):
    if not isinstance(text, str):
        return ""
    # 한자 제거
    text = re.sub(r"[\u4E00-\u9FFF\u3400-\u4DBF\uF900-\uFAFF]", "", text)
    # 양쪽 공백 정리
    return text.strip()

# 한자 제거 적용
df_unique_term["term"] = df_unique_term["term"].apply(remove_hanja)
df_unique_term["description"] = df_unique_term["description"].apply(remove_hanja)

In [43]:
# 한자 제거 후 () 내 빈칸이라면 제거
def remove_empty_parentheses(text):
    if not isinstance(text, str):
        return ""
    # 괄호 안이 완전히 공백이 된 경우 → 괄호 자체 제거
    text = re.sub(r"\(\s*\)", "", text)

    # 괄호 안에 기호만 있는 경우 → 괄호 자체 제거
    text = re.sub(r"\([\s\W]*\)", "", text)

    # 괄호 안에 한글이 있는데 괄호 앞 뒤로는 공백인 경우 → 괄호만 제거
    # (비삼망)은 -> 비삼망
    text = re.sub(r"\(([가-힣]+)\)", r"\1", text)
    
    # 양쪽 공백 정리
    return text.strip()
df_unique_term["term"] = df_unique_term["term"].apply(remove_empty_parentheses)
df_unique_term["description"] = df_unique_term["description"].apply(remove_empty_parentheses)

In [44]:
df_unique_term.to_csv("output/result_no_hanja.csv", index=False)

In [47]:
# term이 빈칸인 행 개수
(df_unique_term["term"].str.strip() == "").sum()

np.int64(57)

In [48]:
# term이 빈칸인 행 제거
df_unique_term = df_unique_term[df_unique_term["term"].str.strip() != ""].reset_index(drop=True)
df_unique_term.to_csv("output/result_no_hanja_ver2.csv", index=False)

In [57]:
# 서술형 뜻풀이 -> 명사형 정리

df_no_hanja = pd.read_csv("output/result_no_hanja_ver2.csv")

def clean_desc(text: str) -> str:
    if not isinstance(text, str):
        return text
    t = text.strip()

    # 1) 흔한 서술형/정의형 어미 제거 (길이가 긴 패턴부터)
    patterns = [
        r"(말하는\s*것입니다?\.)?$",
        r"(말하는\s*것입니다?)?$",
        r"(말하는\s*것\.)?$",
        r"(말하는\s*것)$",
        r"(말합니다?\.)?$",
        r"(말합니다?)?$",
        r"(말이다\.)?$",
        r"(말이다)$",
        r"(말이다)$",
        r"(뜻이다\.)?$",
        r"(뜻이다)$",
        r"(뜻)$",
        r"(뜻을 말한다\.)?$",
        r"(뜻을 말한다)$",
        r"(뜻한다\.)?$",
        r"(뜻한다)$",
        r"(구분이다\.)?$",
        r"(구분이다)$",
        r"(단계이다\.)?$",
        r"(단계이다)$",
        r"(단계이다)$",
        r"(단위이다\.)?$",
        r"(단위이다)$",
        r"(표기이다\.)?$",
        r"(표기이다)$",
        r"(것이다\.)?$",
        r"(것이다)$",
        r"(하였다\.)?$",
        r"(하였다)$",
        r"(했다\.)?$",
        r"(했다)$",
        r"(였다\.)?$",
        r"(였다)$",
        r"(이다\.)?$",
        r"(이다)$",
        r"(다\.)$",
    ]

    for p in patterns:
        t = re.sub(p, "", t).strip()

    # 2) 끝의 불필요한 마침표/쉼표 제거
    t = re.sub(r"[\.·,]+$", "", t).strip()

    return t

df_no_hanja["desc_norm"] = df_no_hanja["description"].astype(str).apply(normalize_desc)
df_no_hanja = df_no_hanja.drop(columns=["description"]).rename(columns={"desc_norm": "description"})

In [58]:
df_no_hanja.to_csv("output/result_no_hanja_ver3.csv", index=False)

In [ ]:
# 번역
# term은 로마자표기법으로 definition은 영어로 번역


In [ ]:
# term 컬럼을 prompt로 변경 : 임진왜란 -> 임진왜란의 뜻을 알려줘
# category가 "인물"인 행 제외
df_filtered = df[df['category'] != "인물"].copy()
df_filtered['prompt'] = df_filtered['term'].apply(lambda x: f"{x}의 뜻을 알려줘")

# category가 "인물"인 행
# 조사 : 선행체언에 따라 다름 - 선행체언이 자음으로 끝날 때 '이' 사용, 모음으로 끝날 때 '가' 사용
df_person = df[df['category'] == "인물"].copy()

def add_particle(name):
    if not name:
        return name
    last_char = name[-1]
    # 한글 자음과 모음의 유니코드 범위
    jongseong_start = 0x11A8
    jongseong_end = 0x11C2
    # 마지막 글자의 종성 추출
    char_code = ord(last_char) - 0xAC00
    jongseong_index = char_code % 28
    if jongseong_index == 0:
        return f"{name}가"
    else:
        return f"{name}이"
    
df_person['prompt'] = df_person['term'].apply(lambda x: f"{add_particle(x)} 누구야?")